In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

print("TensorFlow version:", tf.__version__)
print("Libraries imported successfully!")

TensorFlow version: 2.21.0
Libraries imported successfully!


In [2]:
# Sample text corpus from the practical PDF
text = """I love deep learning.Deep learning is amazing and
powerful.Machine learning and artifical intelligence and exicting
fields.Recurrent Neural Networks(RNNs)are well-suited
for tasks involving sequential data,like text generation"""

# Convert each character into a numerical ID
tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts([text])

total_chars = len(tokenizer.word_index) + 1

# Convert the text into a sequence of integers
sequences = tokenizer.texts_to_sequences([text])[0]

print("Total unique characters:", total_chars - 1)
print("First 20 character IDs:", sequences[:20])

Total unique characters: 30
First 20 character IDs: [4, 3, 6, 12, 16, 1, 3, 9, 1, 1, 17, 3, 6, 1, 5, 7, 2, 4, 2, 10]


In [3]:
sequence_length = 10

def create_sequences(seq, length):
    inputs = []
    targets = []

    for i in range(len(seq) - length):
        inputs.append(seq[i:i + length])
        targets.append(seq[i + 1:i + length + 1])

    return np.array(inputs), np.array(targets)

X, y = create_sequences(sequences, sequence_length)

# One-hot encode target characters
y = to_categorical(y, num_classes=total_chars)

print("Input shape:", X.shape)
print("Target shape:", y.shape)

Input shape: (216, 10)
Target shape: (216, 10, 31)


In [4]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(sequence_length,)),
    tf.keras.layers.Embedding(
        input_dim=total_chars,
        output_dim=50
    ),
    tf.keras.layers.SimpleRNN(
        units=100,
        return_sequences=True
    ),
    tf.keras.layers.Dense(
        total_chars,
        activation="softmax"
    )
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 10, 50)         │         1,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 10, 100)        │        15,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10, 31)         │         3,131 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,781 (77.27 KB)

 Trainable params: 19,781 (77.27 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(
    X,
    y,
    epochs=20,
    batch_size=32,
    verbose=1
)

print("RNN training completed successfully!")

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.1056 - loss: 3.3808
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1731 - loss: 3.1808 
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1866 - loss: 3.0025 
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2514 - loss: 2.8766 
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2995 - loss: 2.7429
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3551 - loss: 2.5862 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3750 - loss: 2.4285
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3972 - loss: 2.2721 
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4259 - loss: 2.1293 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4505 - loss: 1.9915 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4926 - loss: 1.8634 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5324 - loss: 1.7415 
Ep

In [7]:
def generate_text(model, tokenizer, seed_text, length=50):
    seed_text = seed_text.lower()

    # Convert the seed into character IDs
    sequence = tokenizer.texts_to_sequences([seed_text])[0]

    if len(sequence) < sequence_length:
        raise ValueError(
            f"Seed text must contain at least {sequence_length} known characters."
        )

    # Keep the last 10 characters for the model input
    sequence = sequence[-sequence_length:]
    result = seed_text

    for _ in range(length):
        # Model input must have shape (1, 10)
        x_input = np.array([sequence])

        # Predict the next character
        predictions = model.predict(x_input, verbose=0)
        predicted_id = int(np.argmax(predictions[0, -1, :]))

        # Convert predicted ID back to a character
        predicted_char = tokenizer.index_word.get(predicted_id, " ")

        result += predicted_char

        # Slide the input window forward by one character
        sequence = sequence[1:] + [predicted_id]

    return result


seed_text = "Deep learning"

generated_text = generate_text(
    model,
    tokenizer,
    seed_text,
    length=50
)

print("Generated text:")
print(generated_text)

Generated text:
deep learning and artifical intelligence and exicting ae tent a
